# Can your model beat the market? A Dixon-Coles baseline

**Football Charts — 93 leagues, results & goal timing.**

Most football datasets stop at the big five leagues and the final score. This one
goes deep (Premier League to the Baltic and Nordic lower tiers) and includes the
**minute of every goal**.

This notebook fits a **Dixon-Coles** baseline and scores it with log-loss. Your job:

- **Round 1 (here):** beat this baseline's log-loss.
- **Round 2 (coming soon at [football-charts.com](https://www.football-charts.com)):**
  beating a baseline is easy. The betting market prices all of these matches and is
  very hard to beat. Submit probabilities for upcoming matches before kickoff and
  they're scored against the closing line — public, timestamped, honestly settled.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import poisson

df = pd.read_csv("football_charts_matches.csv")
print(df.shape)
df.head()

(77018, 16)


,league,league_name,country,season,match_date,time,home_team,away_team,ht_result,ft_result,home_goals,away_goals,total_goals,result,first_goal_time,all_goal_times
0,algir1,Ligue 1,Algeria,2023-2024,2023-09-15,17:00:00,Khenchela,ES Setif,0:0,1:0,1,0,1,H,61.0,61
1,algir1,Ligue 1,Algeria,2023-2024,2023-09-15,20:00:00,ASO Chlef,Oran,0:0,2:0,2,0,2,H,47.0,"47,62"
2,algir1,Ligue 1,Algeria,2023-2024,2023-09-15,17:00:00,El Bayadh,Constantine,1:0,1:0,1,0,1,H,41.0,41
3,algir1,Ligue 1,Algeria,2023-2024,2023-09-16,17:00:00,Magra,Kabylie,0:1,0:1,0,1,1,A,44.0,44
4,algir1,Ligue 1,Algeria,2023-2024,2023-09-16,17:00:00,MC Alger,Ben Aknoun,2:0,4:0,4,0,4,H,11.0,"11,24,71,90"


### What's in the data

Results, half/full-time scores, and goal-timing columns. **No odds** — results are the ground truth; the market is the Round 2 test.

In [2]:
print("Leagues:", df.league.nunique(), "| Seasons:", sorted(df.season.unique(), key=lambda s:int(str(s).split('-')[0])))
print("\nResult split:")
print(df.result.value_counts(normalize=True).round(3))
df[["league_name","season","home_team","away_team","ft_result","first_goal_time","all_goal_times"]].head()

Leagues: 91 | Seasons: ['2022-2023', '2023-2024', '2023', '2024-2025', '2024', '2025-2026', '2025', '2026']

Result split:
result
H    0.434
A    0.308
D    0.258
Name: proportion, dtype: float64


,league_name,season,home_team,away_team,ft_result,first_goal_time,all_goal_times
0,Ligue 1,2023-2024,Khenchela,ES Setif,1:0,61.0,61
1,Ligue 1,2023-2024,ASO Chlef,Oran,2:0,47.0,"47,62"
2,Ligue 1,2023-2024,El Bayadh,Constantine,1:0,41.0,41
3,Ligue 1,2023-2024,Magra,Kabylie,0:1,44.0,44
4,Ligue 1,2023-2024,MC Alger,Ben Aknoun,4:0,11.0,"11,24,71,90"


### The Dixon-Coles model

Each team gets an **attack** and **defence** strength; there's a **home advantage**
term and a low-score correlation correction **(rho)**. Goals are Poisson; we fit by
maximum likelihood per league on the earlier seasons and test on the latest.

In [3]:
MAX_GOALS = 10

def tau(hg, ag, lh, la, rho):
    out = np.ones_like(hg, dtype=float)
    m00=(hg==0)&(ag==0); m01=(hg==0)&(ag==1); m10=(hg==1)&(ag==0); m11=(hg==1)&(ag==1)
    out[m00] = 1 - lh[m00]*la[m00]*rho
    out[m01] = 1 + lh[m01]*rho
    out[m10] = 1 + la[m10]*rho
    out[m11] = 1 - rho
    return out

def fit_league(train):
    teams = sorted(set(train.home_team) | set(train.away_team))
    idx = {t:i for i,t in enumerate(teams)}; n = len(teams)
    hg, ag = train.home_goals.to_numpy(), train.away_goals.to_numpy()
    hi = train.home_team.map(idx).to_numpy(); ai = train.away_team.map(idx).to_numpy()
    def negll(p):
        atk, dfn, home, rho = p[:n], p[n:2*n], p[2*n], p[2*n+1]
        lh = np.exp(atk[hi]-dfn[ai]+home); la = np.exp(atk[ai]-dfn[hi])
        ll = poisson.logpmf(hg,lh)+poisson.logpmf(ag,la)+np.log(np.clip(tau(hg,ag,lh,la,rho),1e-9,None))
        return -ll.sum()
    p0 = np.concatenate([np.zeros(2*n),[0.25,-0.05]])
    cons = {"type":"eq","fun":lambda p: p[:n].sum()}
    res = minimize(negll, p0, constraints=cons, method="SLSQP", options={"maxiter":200,"ftol":1e-6})
    return idx, res.x[:n], res.x[n:2*n], res.x[2*n], res.x[2*n+1]

def match_1x2(idx, atk, dfn, home_adv, rho, h, a):
    if h not in idx or a not in idx: return None
    lh = np.exp(atk[idx[h]]-dfn[idx[a]]+home_adv); la = np.exp(atk[idx[a]]-dfn[idx[h]])
    m = np.outer(poisson.pmf(np.arange(MAX_GOALS+1),lh), poisson.pmf(np.arange(MAX_GOALS+1),la))
    m[0,0]*=1-lh*la*rho; m[0,1]*=1+lh*rho; m[1,0]*=1+la*rho; m[1,1]*=1-rho
    m/=m.sum()
    return np.tril(m,-1).sum(), np.trace(m), np.triu(m,1).sum()

### Evaluate: per-league holdout

Seasons aren't aligned across leagues (summer `2025` vs winter `2024-2025`), so we hold out each league's most recent season and train on its earlier ones.

In [4]:
skey = lambda s:int(str(s).split("-")[0])
y_true, probs = [], []
for lg, g in df.groupby("league"):
    seasons = sorted(g.season.unique(), key=skey)
    if len(seasons) < 2: continue
    test_season = seasons[-1]
    train, test = g[g.season!=test_season], g[g.season==test_season]
    if len(train) < 100 or len(test) < 20: continue
    try: model = fit_league(train)
    except Exception: continue
    for _, r in test.iterrows():
        p = match_1x2(*model, r.home_team, r.away_team)
        if p is None: continue
        probs.append(p); y_true.append({"H":0,"D":1,"A":2}[r.result])

probs = np.clip(np.array(probs),1e-6,1); probs/=probs.sum(1,keepdims=True)
y = np.array(y_true)
logloss = -np.mean(np.log(probs[np.arange(len(y)), y]))
rates = np.bincount(y,minlength=3)/len(y); naive = -np.mean(np.log(rates[y]))
print(f"Matches scored : {len(y):,}")
print(f"Dixon-Coles log-loss : {logloss:.4f}")
print(f"Naive base-rate      : {naive:.4f}")

Matches scored : 17,709
Dixon-Coles log-loss : 1.0481
Naive base-rate      : 1.0750


## Your turn

**Round 1:** beat the Dixon-Coles log-loss above. Add form, rest days, the goal-timing
signal, xG if you have it, a GBM, whatever.

**Round 2:** think you're good? The market is the real opponent. Submit forward
probabilities at **[football-charts.com](https://www.football-charts.com)**
and see if you beat the closing line. No tips, no claims — just receipts.